# Email Builder Test Notebook

Day 6 이메일 템플릿 설계 및 구현 테스트(EmainBuilder)

## 목차
1. [Setup & Imports](#setup)
2. [EmailBuilder 초기화](#init)
3. [샘플 데이터 준비](#sample-data)
4. [동작 프로세스 테스트](#process-test)
   - Step 1: 상위 아티클 선택 (`_select_top_articles`)
   - Step 2: 카테고리 그룹화 (`_group_by_category`)
   - Step 3: 아티클 포맷팅 (`_format_article`)
   - Step 4: HTML 렌더링 (`render_template`)
5. [완전한 이메일 생성](#full-email)
6. [엣지 케이스 테스트](#edge-cases)
7. [HTML 미리보기](#preview)

In [1]:
import sys
from pathlib import Path
from datetime import datetime, UTC
from uuid import uuid4

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from app.email.builder import EmailBuilder, build_daily_digest_email
from app.db.models import CollectedArticle

print("✅ Imports successful")
print(f"📁 Project root: {project_root}")

✅ Imports successful
📁 Project root: /mnt/d/project/research-curator


### 1. EmailBuilder 초기화

In [2]:
# Initialize EmailBuilder
builder = EmailBuilder()

# Check if template exists
template_dir = Path(project_root) / "src" / "app" / "email" / "templates"
template_file = template_dir / "daily_digest.html"

print(f"📂 Template directory: {template_dir}")
print(f"📄 Template file exists: {template_file.exists()}")

if template_file.exists():
    print(f"✅ Template file found: {template_file.name}")
else:
    print("❌ Template file not found!")

print(f"\n🔧 Jinja2 Environment:")
print(f"   - Loader: {builder.env.loader}")
print(f"   - Autoescape: {builder.env.autoescape}")

📂 Template directory: /mnt/d/project/research-curator/src/app/email/templates
📄 Template file exists: True
✅ Template file found: daily_digest.html

🔧 Jinja2 Environment:
   - Loader: <jinja2.loaders.FileSystemLoader object at 0x7f2a34d0a750>
   - Autoescape: <function select_autoescape.<locals>.autoescape at 0x7f2a1fef1080>


### 2. 샘플 데이터 준비

다양한 importance_score와 source_type을 가진 샘플 아티클 생성

In [3]:
# Create sample articles with different importance scores and types
sample_articles = [
    # Papers (high importance)
    CollectedArticle(
        id=str(uuid4()),
        title="Attention Is All You Need",
        content="We propose a new simple network architecture, the Transformer, based solely on attention mechanisms...",
        summary="트랜스포머 아키텍처를 제안하는 획기적인 논문. 어텐션 메커니즘만으로 시퀀스 모델링을 수행합니다.",
        source_url="https://arxiv.org/abs/1706.03762",
        source_type="paper",
        importance_score=0.95,
        collected_at=datetime.now(UTC),
        article_metadata={
            "authors": ["Ashish Vaswani", "Noam Shazeer", "Niki Parmar", "Jakob Uszkoreit"],
            "citations": 50000,
        },
    ),
    CollectedArticle(
        id=str(uuid4()),
        title="BERT: Pre-training of Deep Bidirectional Transformers",
        content="We introduce a new language representation model called BERT...",
        summary="양방향 트랜스포머를 사용한 사전학습 언어 모델 BERT를 소개합니다.",
        source_url="https://arxiv.org/abs/1810.04805",
        source_type="paper",
        importance_score=0.92,
        collected_at=datetime.now(UTC),
        article_metadata={
            "authors": ["Jacob Devlin", "Ming-Wei Chang", "Kenton Lee"],
            "citations": 45000,
        },
    ),
    # Papers (medium importance)
    CollectedArticle(
        id=str(uuid4()),
        title="LoRA: Low-Rank Adaptation of Large Language Models",
        content="We propose LoRA, which freezes the pre-trained model weights...",
        summary="대규모 언어 모델을 효율적으로 파인튜닝하는 LoRA 기법을 제안합니다.",
        source_url="https://arxiv.org/abs/2106.09685",
        source_type="paper",
        importance_score=0.75,
        collected_at=datetime.now(UTC),
        article_metadata={
            "authors": ["Edward Hu", "Yelong Shen", "Phillip Wallis"],
            "citations": 2000,
        },
    ),
    # News (high importance)
    CollectedArticle(
        id=str(uuid4()),
        title="OpenAI Announces GPT-5: Breaking New Ground in AI",
        content="OpenAI today announced GPT-5, the latest iteration of its groundbreaking language model...",
        summary="OpenAI가 GPT-5를 발표했습니다. 이전 모델보다 10배 향상된 성능을 보입니다.",
        source_url="https://techcrunch.com/gpt5-announcement",
        source_type="news",
        importance_score=0.88,
        collected_at=datetime.now(UTC),
        article_metadata={
            "source": "TechCrunch",
        },
    ),
    CollectedArticle(
        id=str(uuid4()),
        title="Google DeepMind Achieves Breakthrough in Protein Folding",
        content="Google DeepMind's AlphaFold 3 has achieved unprecedented accuracy...",
        summary="구글 딥마인드의 AlphaFold 3가 단백질 구조 예측에서 획기적인 성과를 달성했습니다.",
        source_url="https://www.nature.com/alphafold3",
        source_type="news",
        importance_score=0.85,
        collected_at=datetime.now(UTC),
        article_metadata={
            "source": "Nature",
        },
    ),
    # News (medium importance)
    CollectedArticle(
        id=str(uuid4()),
        title="AI Startups Raise Record $50B in Funding This Year",
        content="AI startups have collectively raised over $50 billion in 2024...",
        summary="올해 AI 스타트업들이 총 500억 달러 이상의 투자를 유치했습니다.",
        source_url="https://venturebeat.com/ai-funding-2024",
        source_type="news",
        importance_score=0.65,
        collected_at=datetime.now(UTC),
        article_metadata={
            "source": "VentureBeat",
        },
    ),
    # Reports (high importance)
    CollectedArticle(
        id=str(uuid4()),
        title="Stanford AI Index Report 2024",
        content="The AI Index 2024 Annual Report tracks, collates, distills, and visualizes data...",
        summary="스탠포드 AI 인덱스 2024 리포트가 발표되었습니다. AI 산업 전반의 동향을 분석합니다.",
        source_url="https://aiindex.stanford.edu/report/",
        source_type="report",
        importance_score=0.82,
        collected_at=datetime.now(UTC),
        article_metadata={
            "organization": "Stanford HAI",
        },
    ),
    # Reports (medium importance)
    CollectedArticle(
        id=str(uuid4()),
        title="McKinsey: The State of AI in 2024",
        content="Our latest survey shows that AI adoption continues to accelerate...",
        summary="맥킨지의 2024 AI 현황 리포트. 기업들의 AI 도입이 가속화되고 있습니다.",
        source_url="https://mckinsey.com/ai-report-2024",
        source_type="report",
        importance_score=0.70,
        collected_at=datetime.now(UTC),
        article_metadata={
            "organization": "McKinsey & Company",
        },
    ),
    # Low importance articles
    CollectedArticle(
        id=str(uuid4()),
        title="Minor Update to PyTorch Documentation",
        content="PyTorch documentation has been updated with minor clarifications...",
        summary="PyTorch 문서가 일부 업데이트되었습니다.",
        source_url="https://pytorch.org/docs",
        source_type="news",
        importance_score=0.45,
        collected_at=datetime.now(UTC),
        article_metadata={
            "source": "PyTorch Blog",
        },
    ),
]

print(f"📊 Created {len(sample_articles)} sample articles")
print(f"\n📈 Importance Score Distribution:")
for article in sorted(sample_articles, key=lambda x: x.importance_score, reverse=True):
    print(f"   {article.importance_score:.2f} - [{article.source_type:6s}] {article.title[:50]}")

📊 Created 9 sample articles

📈 Importance Score Distribution:
   0.95 - [paper ] Attention Is All You Need
   0.92 - [paper ] BERT: Pre-training of Deep Bidirectional Transform
   0.88 - [news  ] OpenAI Announces GPT-5: Breaking New Ground in AI
   0.85 - [news  ] Google DeepMind Achieves Breakthrough in Protein F
   0.82 - [report] Stanford AI Index Report 2024
   0.75 - [paper ] LoRA: Low-Rank Adaptation of Large Language Models
   0.70 - [report] McKinsey: The State of AI in 2024
   0.65 - [news  ] AI Startups Raise Record $50B in Funding This Year
   0.45 - [news  ] Minor Update to PyTorch Documentation


### 3. 동작 프로세스 테스트
#### 3.1 상위 아티클 선택 (`_select_top_articles`)

In [4]:
# Test: Select top 5 articles
limit = 5
top_articles = builder._select_top_articles(sample_articles, limit)

print(f"🔝 Top {limit} articles selected (sorted by importance_score):")
print(f"\n{'Rank':<6} {'Score':<8} {'Type':<8} {'Title'}")
print("=" * 80)

for idx, article in enumerate(top_articles, 1):
    print(f"{idx:<6} {article.importance_score:.2f}     {article.source_type:<8} {article.title[:50]}")

print(f"\n✅ Selected {len(top_articles)} out of {len(sample_articles)} articles")

# Verify sorting
scores = [a.importance_score for a in top_articles]
is_sorted = all(scores[i] >= scores[i+1] for i in range(len(scores)-1))
print(f"✅ Properly sorted by importance: {is_sorted}")

🔝 Top 5 articles selected (sorted by importance_score):

Rank   Score    Type     Title
1      0.95     paper    Attention Is All You Need
2      0.92     paper    BERT: Pre-training of Deep Bidirectional Transform
3      0.88     news     OpenAI Announces GPT-5: Breaking New Ground in AI
4      0.85     news     Google DeepMind Achieves Breakthrough in Protein F
5      0.82     report   Stanford AI Index Report 2024

✅ Selected 5 out of 9 articles
✅ Properly sorted by importance: True


#### 3.2 카테고리 그룹화 (`_group_by_category`)

In [5]:
# Test: Group articles by category
papers, news, reports = builder._group_by_category(top_articles)

print("📚 Category Distribution:")
print(f"\n{'Category':<12} {'Count':<8} {'Articles'}")
print("=" * 80)

print(f"{'Papers':<12} {len(papers):<8}")
for p in papers:
    print(f"{'':12} {'':8} - {p.title[:60]}")

print(f"\n{'News':<12} {len(news):<8}")
for n in news:
    print(f"{'':12} {'':8} - {n.title[:60]}")

print(f"\n{'Reports':<12} {len(reports):<8}")
for r in reports:
    print(f"{'':12} {'':8} - {r.title[:60]}")

total = len(papers) + len(news) + len(reports)
print(f"\n✅ Total grouped articles: {total}")
print(f"✅ Match original count: {total == len(top_articles)}")

📚 Category Distribution:

Category     Count    Articles
Papers       2       
                      - Attention Is All You Need
                      - BERT: Pre-training of Deep Bidirectional Transformers

News         2       
                      - OpenAI Announces GPT-5: Breaking New Ground in AI
                      - Google DeepMind Achieves Breakthrough in Protein Folding

Reports      1       
                      - Stanford AI Index Report 2024

✅ Total grouped articles: 5
✅ Match original count: True


#### 3.3 아티클 포맷팅 (`_format_article`)

In [6]:
# Test: Format articles for template
print("🎨 Formatted Articles:")
print("\n" + "=" * 80)

for article in top_articles[:3]:  # Show first 3 for brevity
    formatted = builder._format_article(article)
    
    print(f"\n📄 Title: {formatted['title']}")
    print(f"   Type: {article.source_type}")
    print(f"   Score: {formatted['importance_score']:.2f}")
    print(f"   Stars: {formatted['importance_stars']}")
    print(f"   Label: {formatted['importance_label']}")
    print(f"   Summary: {formatted['summary'][:100]}...")
    print(f"   URL: {formatted['source_url']}")
    print(f"   Date: {formatted['published_date']}")
    
    # Show type-specific metadata
    if article.source_type == "paper":
        print(f"   Authors: {formatted.get('authors', 'N/A')}")
        print(f"   Citations: {formatted.get('citations', 'N/A')}")
    elif article.source_type == "news":
        print(f"   Source: {formatted.get('source', 'N/A')}")
    elif article.source_type == "report":
        print(f"   Organization: {formatted.get('organization', 'N/A')}")
    
    print("=" * 80)

print("\n✅ Article formatting test complete")

🎨 Formatted Articles:


📄 Title: Attention Is All You Need
   Type: paper
   Score: 0.95
   Stars: ⭐⭐⭐
   Label: 높음
   Summary: 트랜스포머 아키텍처를 제안하는 획기적인 논문. 어텐션 메커니즘만으로 시퀀스 모델링을 수행합니다....
   URL: https://arxiv.org/abs/1706.03762
   Date: 2025-12-18
   Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar 외 1명
   Citations: 50000

📄 Title: BERT: Pre-training of Deep Bidirectional Transformers
   Type: paper
   Score: 0.92
   Stars: ⭐⭐⭐
   Label: 높음
   Summary: 양방향 트랜스포머를 사용한 사전학습 언어 모델 BERT를 소개합니다....
   URL: https://arxiv.org/abs/1810.04805
   Date: 2025-12-18
   Authors: Jacob Devlin, Ming-Wei Chang, Kenton Lee
   Citations: 45000

📄 Title: OpenAI Announces GPT-5: Breaking New Ground in AI
   Type: news
   Score: 0.88
   Stars: ⭐⭐⭐
   Label: 높음
   Summary: OpenAI가 GPT-5를 발표했습니다. 이전 모델보다 10배 향상된 성능을 보입니다....
   URL: https://techcrunch.com/gpt5-announcement
   Date: 2025-12-18
   Source: TechCrunch

✅ Article formatting test complete


#### 3.4-1 Importance Level 테스트

In [7]:
# Test importance level calculation
print("⭐ Importance Level Test:\n")

test_scores = [
    (0.95, "high", "⭐⭐⭐", "높음"),
    (0.80, "high", "⭐⭐⭐", "높음"),
    (0.75, "medium", "⭐⭐", "중간"),
    (0.60, "medium", "⭐⭐", "중간"),
    (0.55, "low", "⭐", "낮음"),
    (0.30, "low", "⭐", "낮음"),
]

print(f"{'Score':<8} {'Expected Level':<16} {'Expected Stars':<16} {'Expected Label':<16} {'Result'}")
print("=" * 80)

all_pass = True
for score, exp_level, exp_stars, exp_label in test_scores:
    # Create test article
    test_article = CollectedArticle(
        id=str(uuid4()),
        title="Test Article",
        content="Test content",
        source_url="https://test.com",
        source_type="paper",
        importance_score=score,
        collected_at=datetime.now(UTC),
    )
    
    formatted = builder._format_article(test_article)
    
    passed = (
        formatted['importance_level'] == exp_level and
        formatted['importance_stars'] == exp_stars and
        formatted['importance_label'] == exp_label
    )
    
    result = "✅ PASS" if passed else "❌ FAIL"
    all_pass = all_pass and passed
    
    print(f"{score:<8} {exp_level:<16} {exp_stars:<16} {exp_label:<16} {result}")

print(f"\n{'✅ All tests passed!' if all_pass else '❌ Some tests failed!'}")

⭐ Importance Level Test:

Score    Expected Level   Expected Stars   Expected Label   Result
0.95     high             ⭐⭐⭐              높음               ✅ PASS
0.8      high             ⭐⭐⭐              높음               ✅ PASS
0.75     medium           ⭐⭐               중간               ✅ PASS
0.6      medium           ⭐⭐               중간               ✅ PASS
0.55     low              ⭐                낮음               ✅ PASS
0.3      low              ⭐                낮음               ✅ PASS

✅ All tests passed!


#### 3.4-2 Authors Formatting 테스트

In [8]:
# Test authors formatting
print("👥 Authors Formatting Test:\n")

test_cases = [
    ([], None),
    (["John Doe"], "John Doe"),
    (["John Doe", "Jane Smith"], "John Doe, Jane Smith"),
    (["A", "B", "C"], "A, B, C"),
    (["A", "B", "C", "D"], "A, B, C 외 1명"),
    (["A", "B", "C", "D", "E"], "A, B, C 외 2명"),
]

print(f"{'Input':<40} {'Expected':<30} {'Actual':<30} {'Result'}")
print("=" * 110)

all_pass = True
for authors, expected in test_cases:
    actual = builder._format_authors(authors)
    passed = actual == expected
    all_pass = all_pass and passed
    
    input_str = str(authors)[:38]
    expected_str = str(expected)[:28]
    actual_str = str(actual)[:28]
    result = "✅" if passed else "❌"
    
    print(f"{input_str:<40} {expected_str:<30} {actual_str:<30} {result}")

print(f"\n{'✅ All tests passed!' if all_pass else '❌ Some tests failed!'}")

👥 Authors Formatting Test:

Input                                    Expected                       Actual                         Result
[]                                       None                           None                           ✅
['John Doe']                             John Doe                       John Doe                       ✅
['John Doe', 'Jane Smith']               John Doe, Jane Smith           John Doe, Jane Smith           ✅
['A', 'B', 'C']                          A, B, C                        A, B, C                        ✅
['A', 'B', 'C', 'D']                     A, B, C 외 1명                   A, B, C 외 1명                   ✅
['A', 'B', 'C', 'D', 'E']                A, B, C 외 2명                   A, B, C 외 2명                   ✅

✅ All tests passed!


#### 3.5 HTML 렌더링 (`render_template`)

In [9]:
# Test: Render minimal template context
test_context = {
    "service_name": "Research Curator",
    "date": "2024년 12월 04일",
    "user_name": "테스트 사용자",
    "user_email": "test@example.com",
    "papers": [],
    "news": [],
    "reports": [],
    "settings_url": "http://localhost:8501/settings",
    "feedback_url": "http://localhost:8501/feedback",
    "unsubscribe_url": "http://localhost:8501/unsubscribe?email=test@example.com",
}

try:
    html = builder.render_template("daily_digest.html", test_context)
    print("✅ Template rendered successfully")
    print(f"\n📏 HTML length: {len(html)} characters")
    print(f"\n📄 First 500 characters of HTML:")
    print("=" * 80)
    print(html[:500])
    print("...")
    print("=" * 80)
except Exception as e:
    print(f"❌ Template rendering failed: {e}")

✅ Template rendered successfully

📏 HTML length: 8514 characters

📄 First 500 characters of HTML:
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <title>Research Curator - 오늘의 AI 연구 동향</title>
    <style>
        /* Reset styles */
        body, table, td, a { -webkit-text-size-adjust: 100%; -ms-text-size-adjust: 100%; }
        table, td { mso-table-lspace: 0pt; mso-table-rspace: 0pt; }
        img { -ms-interpolation-mode: bicubic; bord
...


### 4. 완전한 이메일 생성 <a name="full-email"></a>

전체 프로세스를 통합하여 실제 이메일 HTML 생성

In [10]:
# Generate complete daily digest email
user_name = "김연구"
user_email = "researcher.kim@example.com"
daily_limit = 5

print(f"📧 Generating daily digest email for {user_name} ({user_email})")
print(f"📊 Daily limit: {daily_limit} articles")
print(f"📚 Total available articles: {len(sample_articles)}")
print("\n" + "=" * 80)

# Build email
html_email = builder.build_daily_digest(
    user_name=user_name,
    user_email=user_email,
    articles=sample_articles,
    daily_limit=daily_limit,
)

print("\n✅ Email generated successfully!")
print(f"\n📏 Total HTML size: {len(html_email):,} characters ({len(html_email) / 1024:.1f} KB)")

# Check for key elements
checks = [
    ("User name" in html_email or user_name in html_email, "User name personalization"),
    ("Research Curator" in html_email, "Service name"),
    ("2024년" in html_email, "Date"),
    ("settings" in html_email.lower(), "Settings link"),
    ("feedback" in html_email.lower(), "Feedback link"),
    ("unsubscribe" in html_email.lower(), "Unsubscribe link"),
]

print("\n🔍 Content Validation:")
for passed, description in checks:
    status = "✅" if passed else "❌"
    print(f"   {status} {description}")

# Count sections
has_papers = "Papers" in html_email or "논문" in html_email
has_news = "News" in html_email or "뉴스" in html_email
has_reports = "Reports" in html_email or "리포트" in html_email

print("\n📑 Sections:")
print(f"   {'✅' if has_papers else '❌'} Papers section")
print(f"   {'✅' if has_news else '❌'} News section")
print(f"   {'✅' if has_reports else '❌'} Reports section")

📧 Generating daily digest email for 김연구 (researcher.kim@example.com)
📊 Daily limit: 5 articles
📚 Total available articles: 9


✅ Email generated successfully!

📏 Total HTML size: 16,587 characters (16.2 KB)

🔍 Content Validation:
   ✅ User name personalization
   ✅ Service name
   ❌ Date
   ✅ Settings link
   ✅ Feedback link
   ✅ Unsubscribe link

📑 Sections:
   ✅ Papers section
   ✅ News section
   ✅ Reports section


### 5. 엣지 케이스 테스트 <a name="edge-cases"></a>

#### Test 1: Empty Articles List

In [11]:
# Test with empty articles
try:
    empty_html = builder.build_daily_digest(
        user_name="Empty User",
        user_email="empty@example.com",
        articles=[],
        daily_limit=5,
    )
    print("✅ Empty articles handled successfully")
    print(f"   HTML size: {len(empty_html)} characters")
    print(f"   Contains user name: {'Empty User' in empty_html}")
except Exception as e:
    print(f"❌ Failed to handle empty articles: {e}")

✅ Empty articles handled successfully
   HTML size: 8519 characters
   Contains user name: True


#### Test 2: Single Category Only

In [12]:
# Test with only papers
papers_only = [a for a in sample_articles if a.source_type == "paper"]

try:
    papers_html = builder.build_daily_digest(
        user_name="Papers Only User",
        user_email="papers@example.com",
        articles=papers_only,
        daily_limit=5,
    )
    print("✅ Single category (papers only) handled successfully")
    print(f"   HTML size: {len(papers_html)} characters")
    
    # Verify only papers section exists
    has_papers = "📚" in papers_html or "Papers" in papers_html
    print(f"   Contains papers section: {has_papers}")
except Exception as e:
    print(f"❌ Failed with single category: {e}")

✅ Single category (papers only) handled successfully
   HTML size: 13189 characters
   Contains papers section: True


#### Test 3: Very Long Summary

In [13]:
# Test summary truncation
long_article = CollectedArticle(
    id=str(uuid4()),
    title="Article with Very Long Summary",
    content="Short content",
    summary="This is a very long summary. " * 50,  # 1400+ characters
    source_url="https://example.com",
    source_type="paper",
    importance_score=0.9,
    collected_at=datetime.now(UTC),
    article_metadata={},
)

formatted_long = builder._format_article(long_article)

print(f"Original summary length: {len(long_article.summary)} characters")
print(f"Formatted summary length: {len(formatted_long['summary'])} characters")
print(f"Truncated: {len(formatted_long['summary']) <= 200}")
print(f"Ends with '...': {formatted_long['summary'].endswith('...')}")

if len(formatted_long['summary']) <= 200:
    print("\n✅ Summary truncation working correctly")
else:
    print("\n❌ Summary truncation not working")

Original summary length: 1450 characters
Formatted summary length: 200 characters
Truncated: True
Ends with '...': True

✅ Summary truncation working correctly


#### Test 4: Convenience Function

In [14]:
# Test convenience function
try:
    convenience_html = build_daily_digest_email(
        user_name="Convenience User",
        user_email="convenience@example.com",
        articles=sample_articles[:3],
        daily_limit=3,
    )
    print("✅ Convenience function works")
    print(f"   HTML size: {len(convenience_html)} characters")
except Exception as e:
    print(f"❌ Convenience function failed: {e}")

✅ Convenience function works
   HTML size: 13199 characters


### 6. HTML 미리보기 <a name="preview"></a>

생성된 HTML을 파일로 저장하고 브라우저에서 미리보기

In [15]:
# Save HTML to file for preview
output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "email_preview.html"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(html_email)

print(f"✅ HTML saved to: {output_file}")
print(f"\n🌐 To preview:")
print(f"   1. Open the file in a browser: {output_file.absolute()}")
print(f"   2. Or run: open {output_file}  (macOS)")
print(f"   3. Or run: start {output_file}  (Windows)")

✅ HTML saved to: /mnt/d/project/research-curator/notebooks/output/email_preview.html

🌐 To preview:
   1. Open the file in a browser: /mnt/d/project/research-curator/notebooks/output/email_preview.html
   2. Or run: open /mnt/d/project/research-curator/notebooks/output/email_preview.html  (macOS)
   3. Or run: start /mnt/d/project/research-curator/notebooks/output/email_preview.html  (Windows)


#### Display HTML in Notebook (IPython)

In [16]:
from IPython.display import HTML, display

# Display the email in notebook
print("📧 Email Preview:")
print("=" * 80)
display(HTML(html_email))

📧 Email Preview:


🔬 Research Curator 📅 2025년 12월 18일
⚙️ 설정 변경 💬 피드백 이 이메일은 researcher.kim@example.com로 발송되었습니다. 수신을 원하지 않으시면 구독 취소를 클릭해주세요. © 2025 Research Curator. All rights reserved.


## 📊 Summary

### EmailBuilder 동작 프로세스 검증 완료

```
입력: 아티클 리스트 (from Vector DB or PostgreSQL)
   ↓
[_select_top_articles] - importance_score 기준 정렬 및 상위 N개 선택 ✅
   ↓
[_group_by_category] - paper/news/report 그룹화 ✅
   ↓
[_format_article] - 각 아티클을 템플릿 포맷으로 변환 ✅
   - 중요도 별 stars 생성 (⭐⭐⭐)
   - 날짜 포맷팅
   - 요약문 길이 제한
   ↓
[render_template] - Jinja2로 HTML 렌더링 ✅
   - 사용자 이름 삽입
   - 날짜 삽입
   - 아티클 섹션 생성
   - Footer 링크 생성
   ↓
출력: 완전한 HTML 이메일 (String) ✅
```

### 테스트 결과

- ✅ 템플릿 초기화 및 로딩
- ✅ 상위 아티클 선택 (importance_score 기준)
- ✅ 카테고리 그룹화 (paper/news/report)
- ✅ 아티클 포맷팅 (stars, 날짜, 요약 등)
- ✅ Importance level 계산 (high/medium/low)
- ✅ Authors 포맷팅 (최대 3명 + "외 N명")
- ✅ HTML 렌더링
- ✅ 완전한 이메일 생성
- ✅ 엣지 케이스 처리 (빈 리스트, 단일 카테고리, 긴 요약)
- ✅ Convenience 함수
- ✅ HTML 파일 저장 및 미리보기

**All tests passed! 🎉**